In [ ]:
# Run once in Colab
!pip install sacrebleu


In [ ]:
# === Setup + imports ===

import torch, random, time, json
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import sacrebleu

# Settings - edit paths if needed
DATA_PATH = "/content/drive/MyDrive/TensorData/padded_dataset.pt"   # change if your path differs
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
BATCH_SIZE = 64
LR = 1e-3
NUM_EPOCHS = 10          # increase for full training
CLIP = 1.0
TRAIN_RATIO = 0.9

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


In [ ]:
# === Utility: load data ===
data = torch.load(DATA_PATH)
src_tensor = data["src_tensor"]         # (N, max_src_len)
src_lengths = data["src_lengths"]
tgt_tensor = data["tgt_tensor"]         # (N, max_tgt_len)
tgt_lengths = data["tgt_lengths"]
vocab_src = data["vocab_src"]           # token -> id
vocab_tgt = data["vocab_tgt"]


In [ ]:

# reverse maps
id2tgt = {i:t for t,i in vocab_tgt.items()}
id2src = {i:t for t,i in vocab_src.items()}

PAD_SRC = vocab_src["<pad>"]
PAD_TGT = vocab_tgt["<pad>"]


In [ ]:
# === Dataset wrapper ===
class ParallelDataset(Dataset):
    def __init__(self, src_tensor, src_lengths, tgt_tensor, tgt_lengths):
        self.src = src_tensor
        self.src_len = src_lengths
        self.tgt = tgt_tensor
        self.tgt_len = tgt_lengths
    def __len__(self):
        return self.src.size(0)
    def __getitem__(self, idx):
        return self.src[idx], self.src_len[idx], self.tgt[idx], self.tgt_len[idx]


In [ ]:
# create train/test split
N = src_tensor.size(0)
idxs = np.arange(N)
np.random.shuffle(idxs)
n_train = int(TRAIN_RATIO * N)
train_idx = idxs[:n_train].tolist()
val_idx = idxs[n_train:].tolist()

train_ds = torch.utils.data.Subset(ParallelDataset(src_tensor, src_lengths, tgt_tensor, tgt_lengths), train_idx)
val_ds   = torch.utils.data.Subset(ParallelDataset(src_tensor, src_lengths, tgt_tensor, tgt_lengths), val_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

print("Train size:", len(train_ds), "Val size:", len(val_ds))


Train size: 18770 Val size: 2086


In [ ]:

# === Model classes (encoder-decoder no attention) ===
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, pad_idx, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers=num_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if num_layers>1 else 0.0
        )
        # project concatenated last bi-hidden states -> decoder hidden size * num_layers
        self.fc_h = nn.Linear(hidden_dim*2, hidden_dim * num_layers)
        self.fc_c = nn.Linear(hidden_dim*2, hidden_dim * num_layers)
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

    def forward(self, src, src_lengths):
        # src: (B, S)
        embedded = self.embedding(src)  # (B, S, E)
        packed = nn.utils.rnn.pack_padded_sequence(embedded, src_lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, (h, c) = self.lstm(packed)
        outputs, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
        # h,c: (num_layers*2, B, hidden_dim)
        # take last forward & backward from top layer: indices -2 and -1
        h_cat = torch.cat((h[-2,:,:], h[-1,:,:]), dim=1)  # (B, hidden_dim*2)
        c_cat = torch.cat((c[-2,:,:], c[-1,:,:]), dim=1)
        # project to (num_layers, B, hidden_dim)
        h0 = self.fc_h(h_cat).view(self.num_layers, -1, self.hidden_dim)
        c0 = self.fc_c(c_cat).view(self.num_layers, -1, self.hidden_dim)
        return outputs, (h0, c0)

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, pad_idx, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers=num_layers,
            batch_first=True, dropout=dropout if num_layers>1 else 0.0
        )
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, tgt, hidden):
        # tgt: (B, T) -> embeddings -> LSTM
        emb = self.embedding(tgt)  # (B, T, E)
        out, hidden = self.lstm(emb, hidden)  # out: (B, T, H)
        logits = self.fc_out(out)  # (B, T, V)
        return logits, hidden

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
    def forward(self, src, src_lengths, tgt_input):
        # src: (B, S), tgt_input: (B, T_in)
        _, hidden = self.encoder(src, src_lengths)
        logits, _ = self.decoder(tgt_input, hidden)  # (B, T_in, V)
        return logits


In [ ]:
# === Freeze helper ===
def freeze_encoder(encoder):
    for p in encoder.parameters():
        p.requires_grad = False

def unfreeze_encoder(encoder):
    for p in encoder.parameters():
        p.requires_grad = True

In [ ]:
# === Greedy decode helper for evaluation (batch) ===
def greedy_decode_batch(model, src_batch, src_lengths, max_len=100):
    # returns list of predicted strings (subword tokens joined)
    model.eval()
    B = src_batch.size(0)
    with torch.no_grad():
        _, hidden = model.encoder(src_batch, src_lengths)  # hidden sized (n_layers, B, H)
        # start tokens: <sos>
        sos = vocab_tgt["<sos>"]
        eos = vocab_tgt["<eos>"]
        # initialize current tokens (B,1)
        cur = torch.full((B,1), sos, dtype=torch.long, device=DEVICE)
        outputs = [[] for _ in range(B)]
        h = hidden
        for step in range(max_len):
            emb = model.decoder.embedding(cur)  # (B,1,E)
            out, h = model.decoder.lstm(emb, h)  # out: (B,1,H)
            logits = model.decoder.fc_out(out)    # (B,1,V)
            next_token = logits.argmax(dim=-1)    # (B,1)
            cur = next_token  # feed next
            for i in range(B):
                tid = int(next_token[i,0].cpu().item())
                if tid == eos:
                    continue
                outputs[i].append(tid)
        # convert ids -> string
        preds = []
        for seq in outputs:
            toks = [ id2tgt.get(i, "<unk>") for i in seq ]
            # combine tokens into a string; tokens include '_' markers
            text = "".join(toks).replace("_", " ").strip()
            preds.append(text)
        return preds

In [ ]:
# === Training / evaluation functions ===
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for batch in loader:
        src_b, src_len_b, tgt_b, _ = batch
        src_b = src_b.to(DEVICE); src_len_b = src_len_b.to(DEVICE)
        tgt_b = tgt_b.to(DEVICE)
        # prepare decoder input / target
        tgt_input = tgt_b[:, :-1]   # includes <sos>
        tgt_out   = tgt_b[:, 1:]    # shifted, includes <eos>

        optimizer.zero_grad()
        logits = model(src_b, src_len_b, tgt_input)   # (B, T, V)
        B,T,V = logits.size()
        loss = criterion(logits.view(-1, V), tgt_out.contiguous().view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_( [p for p in model.parameters() if p.requires_grad], CLIP)
        optimizer.step()
        total_loss += loss.item() * src_b.size(0)
    return total_loss / len(loader.dataset)


In [ ]:
def evaluate_model(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_refs = []
    all_hyps = []
    with torch.no_grad():
        for batch in loader:
            src_b, src_len_b, tgt_b, _ = batch
            src_b = src_b.to(DEVICE); src_len_b = src_len_b.to(DEVICE)
            tgt_b = tgt_b.to(DEVICE)
            tgt_input = tgt_b[:, :-1]
            tgt_out = tgt_b[:, 1:]
            logits = model(src_b, src_len_b, tgt_input)
            B,T,V = logits.size()
            loss = criterion(logits.view(-1, V), tgt_out.contiguous().view(-1))
            total_loss += loss.item() * src_b.size(0)

            # greedy decode batch for BLEU
            hyps = greedy_decode_batch(model, src_b, src_len_b, max_len=tgt_b.size(1))
            # references: convert tgt_b to text
            for i in range(tgt_b.size(0)):
                # target tokens ids excluding <sos> and after > drop <sos>
                ids = tgt_b[i].cpu().numpy().tolist()[1:]  # drop <sos>, keep up to <eos>
                # convert until <eos>
                toks = []
                for tid in ids:
                    if tid == vocab_tgt["<eos>"]: break
                    toks.append(id2tgt.get(tid, "<unk>"))
                ref_text = "".join(toks).replace("_", " ").strip()
                all_refs.append(ref_text)
            all_hyps.extend(hyps)

    # compute BLEU
    bleu = sacrebleu.corpus_bleu(all_hyps, [all_refs])
    return total_loss / len(loader.dataset), bleu.score, all_refs[:5], all_hyps[:5]

# === Run two experiments: A (freeze encoder) then B (train whole) ===
results = {}

In [ ]:

for exp_name, freeze_enc in [("freeze_encoder", True), ("train_both", False)]:
    print("\n\n=== Experiment:", exp_name, " freeze_encoder=", freeze_enc, " ===")
    # instantiate fresh models for each experiment
    EMBED_DIM = 512
    HIDDEN_DIM = 512
    ENC_LAYERS = 2
    DEC_LAYERS = 2
    encoder = Encoder(len(vocab_src), EMBED_DIM, HIDDEN_DIM, ENC_LAYERS, pad_idx=PAD_SRC).to(DEVICE)
    decoder = Decoder(len(vocab_tgt), EMBED_DIM, HIDDEN_DIM, DEC_LAYERS, pad_idx=PAD_TGT).to(DEVICE)
    model = Seq2Seq(encoder, decoder, DEVICE).to(DEVICE)

    # Optionally initialize embeddings from precomputed embedding matrices if you have them:
    # encoder.embedding.weight.data.copy_(pretrained_src_embeddings)
    # decoder.embedding.weight.data.copy_(pretrained_tgt_embeddings)

    if freeze_enc:
        freeze_encoder(model.encoder)
    else:
        pass  # keep trainable

    # optimizer: only parameters with requires_grad=True
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(trainable_params, lr=LR)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_TGT)

    # training loop
    best_val_bleu = -1.0
    history = []
    for epoch in range(1, NUM_EPOCHS+1):
        t0 = time.time()
        train_loss = train_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_bleu, sample_refs, sample_hyps = evaluate_model(model, val_loader, criterion)
        elapsed = time.time() - t0
        print(f"[{exp_name}] Epoch {epoch} time {elapsed:.1f}s train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_bleu={val_bleu:.2f}")
        history.append((epoch, train_loss, val_loss, val_bleu))
        # save best
        if val_bleu > best_val_bleu:
            best_val_bleu = val_bleu
            torch.save(model.state_dict(), f"best_{exp_name}.pt")
            print("Saved best model", f"best_{exp_name}.pt")
    results[exp_name] = {"history": history, "best_val_bleu": best_val_bleu,
                         "sample_refs": sample_refs, "sample_hyps": sample_hyps}




=== Experiment: freeze_encoder  freeze_encoder= True  ===
[freeze_encoder] Epoch 1 time 706.2s train_loss=4.4116 val_loss=3.8768 val_bleu=0.03
Saved best model best_freeze_encoder.pt
[freeze_encoder] Epoch 2 time 698.7s train_loss=3.6907 val_loss=3.5563 val_bleu=0.03
[freeze_encoder] Epoch 3 time 696.8s train_loss=3.4396 val_loss=3.4094 val_bleu=0.02
[freeze_encoder] Epoch 4 time 696.9s train_loss=3.2862 val_loss=3.3301 val_bleu=0.03
[freeze_encoder] Epoch 5 time 698.5s train_loss=3.1707 val_loss=3.2898 val_bleu=0.04
Saved best model best_freeze_encoder.pt


In [ ]:

# Print summary
print("\n=== Summary ===")
for k,v in results.items():
    print(k, "best BLEU:", v["best_val_bleu"])
    print("sample refs:", v["sample_refs"])
    print("sample hyps:", v["sample_hyps"])



=== Summary ===
freeze_encoder best BLEU: 0.20835056075137995
sample refs: ['koi to harf lab-e-charagar se nikla tha', 'maana ki us se milna milana bahut hua', 'us chashm-e-fusun-gar ka agar paae ishara', 'vo meri raton ko chhup kar guzarne vaala', 'bad-gumani ne na chaha use sargarm-e-khiram']
sample hyps: ['vo bhi to khush raha huun main shaeri hai hai hai hai', 'vo bhi to khush raha huun main shaeri hai hai hai hai', 'vo bhi to khush raha huun main shaeri hai hai hai hai', 'vo bhi to khush raha huun main shaeri hai hai hai hai', 'vo bhi to khush raha huun main shaeri hai hai hai hai']
train_both best BLEU: 0.07814578730311522
sample refs: ['koi to harf lab-e-charagar se nikla tha', 'maana ki us se milna milana bahut hua', 'us chashm-e-fusun-gar ka agar paae ishara', 'vo meri raton ko chhup kar guzarne vaala', 'bad-gumani ne na chaha use sargarm-e-khiram']
sample hyps: ['sham-e-gham ka haq hai aur main huun shakhs allah gard gard par gard ne sham-e-kusha', 'shokhi-e-raftar se bahar 